# 07 — Régression multiple : pace par-sortie expliquée par le contexte et la forme

**Question :** à conditions et forme physique données, mes courses sont-elles reproductibles ?

**Approche :** régression linéaire multiple par-sortie, avec sélection de variables par AIC.

**Variables candidates :**
- *Contexte météo* : T°, T², humidité
- *Contexte effort* : distance, D+/km
- *Forme physique* : pace moyenne des 5 dernières sorties (feature clé), volumes glissants 7/14/28/90j, ACWR, monotonie, days_since_last, days_since_last_fast

**Grain d'analyse :** 1 sortie = 1 ligne (préférence M. Quost). La question TZ enregistrée porte sur l'allure *instantanée* ; cette approche agrégée en est un sous-problème. Le modèle mixte par-km (notebook 09) traitera la question complète.

## 1. Setup et chargement de la table enrichie

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import sys
sys.path.append("../streamlit_app")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import f as f_dist
from sqlalchemy import create_engine

plt.rcParams["figure.figsize"] = (10, 5)

# Chargement direct depuis activities_enriched (voir src/enrich_features.py)
engine = create_engine(os.environ["DATABASE_URL"].replace("postgres://", "postgresql://", 1))
df = pd.read_sql("SELECT * FROM activities_enriched", engine)
df["date_only"] = pd.to_datetime(df["start_date_local"]).dt.tz_localize(None).dt.floor("D")
print(f"df : {len(df)} sorties, {len(df.columns)} colonnes")

## 2. Préparation de l'échantillon de modélisation

In [ ]:
# Filtres : exclut micro-sorties (<1 km) et longs trails (>=30 km)
dfm = df[(df["distance_km"] >= 1) & (df["distance_km"] < 30)].copy()

MODEL_VARS = ["pace_min_km", "weather_temperature", "weather_humidity",
              "distance_km", "elev_per_km",
              "vol_7d_km", "vol_14d_km", "vol_28d_km", "vol_90d_km",
              "acwr_7_28", "monotony_28d", "days_since_last",
              "pace_last_5_mean", "days_since_last_fast"]

print(f"n après filtres distance : {len(dfm)}")
print("\nNA par variable :")
print(dfm[MODEL_VARS].isna().sum())
print(f"\nn exploitable (sans NA sur pace_last_5_mean) : {len(dfm.dropna(subset=['pace_last_5_mean']))}")

## 3. Sélection de variables : effet marginal de chaque feature de forme

On fit un modèle baseline (météo + effort) puis on ajoute **une feature de forme à la fois** pour mesurer son apport marginal propre.

In [ ]:
FORME_FEATURES = ["vol_7d_km", "vol_14d_km", "vol_28d_km", "vol_90d_km",
                  "n_runs_7d", "n_runs_14d", "n_runs_28d",
                  "acwr_7_28", "monotony_28d",
                  "days_since_last", "pace_last_5_mean", "days_since_last_fast"]

f_base = ("pace_min_km ~ weather_temperature + I(weather_temperature**2) "
          "+ distance_km + elev_per_km")
base = smf.ols(f_base, data=dfm).fit()
print(f"Baseline (4 var) : R² adj = {base.rsquared_adj:.3f}  AIC = {base.aic:.1f}  n = {int(base.nobs)}")

rows = []
for f in FORME_FEATURES:
    sub = dfm.dropna(subset=[f])
    try:
        m = smf.ols(f_base + f" + {f}", data=sub).fit()
        rows.append({"feature": f, "R²_adj": m.rsquared_adj,
                     "ΔR²_adj": m.rsquared_adj - base.rsquared_adj,
                     "ΔAIC": m.aic - base.aic,
                     "p_value": m.pvalues[f], "n": int(m.nobs)})
    except Exception as e:
        rows.append({"feature": f, "erreur": str(e)})

comparison = pd.DataFrame(rows).sort_values("ΔAIC")
print("\nAjout individuel de chaque feature de forme à la baseline :")
print(comparison.to_string(index=False))
print("\n→ pace_last_5_mean domine largement : seule feature avec gain réel de R² adj et AIC fortement amélioré.")

## 4. Modèle final retenu

**Variables :** T° + T² + distance + D+/km + pace_last_5_mean.

Humidité, volumes glissants, ACWR, monotonie, days_since_last* : retirés car non significatifs individuellement et modèle plus parcimonieux sans eux.

In [ ]:
FORMULA = ("pace_min_km ~ weather_temperature + I(weather_temperature**2) "
           "+ distance_km + elev_per_km + pace_last_5_mean")

dfm = dfm.dropna(subset=["pace_last_5_mean"]).reset_index(drop=True)
model = smf.ols(FORMULA, data=dfm).fit()
print(model.summary())

In [ ]:
# Chiffres clés
print("=== Modèle final ===")
print(f"n = {int(model.nobs)}")
print(f"R² = {model.rsquared:.3f}, R² ajusté = {model.rsquared_adj:.3f}")
print(f"RMSE = {np.sqrt(model.mse_resid):.3f} min/km = {np.sqrt(model.mse_resid)*60:.1f} s/km")
print(f"Durbin-Watson = {durbin_watson(model.resid):.3f}  (2 = pas d'autocorrélation)")
print(f"F = {model.fvalue:.2f}, p = {model.f_pvalue:.2e}")

## 5. Diagnostics de colinéarité (VIF)

In [ ]:
X = dfm[["weather_temperature", "distance_km", "elev_per_km", "pace_last_5_mean"]].copy()
X["T2"] = X["weather_temperature"]**2
X = sm.add_constant(X)

vif = pd.DataFrame({"var": X.columns,
                    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
print(vif.round(2))
print("\n→ VIF élevés attendus pour T° et T² (colinéaires par construction du polynôme).")
print("→ VIF des autres prédicteurs < 2 : pas de problème.")

## 6. Diagnostics des résidus

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sm.qqplot(model.resid, line="45", fit=True, ax=axes[0])
axes[0].set_title("QQ-plot des résidus")

axes[1].scatter(model.fittedvalues, model.resid, alpha=0.6, s=20, color="#FC4C02")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("Pace prédite (min/km)")
axes[1].set_ylabel("Résidus")
axes[1].set_title("Résidus vs valeurs ajustées")

dfm_fit = dfm.copy()
dfm_fit["resid"] = model.resid.values
dfm_fit_sorted = dfm_fit.sort_values("date_only")
axes[2].plot(dfm_fit_sorted["date_only"], dfm_fit_sorted["resid"], "o-",
             alpha=0.6, markersize=4, color="#FC4C02")
axes[2].axhline(0, color="black", lw=1)
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Résidus")
axes[2].set_title("Résidus dans l'ordre chronologique")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 7. Observations influentes (Cook's distance)

In [ ]:
infl = OLSInfluence(model)
cooks_d = infl.cooks_distance[0]
seuil = 4 / len(dfm)

dfm_fit["cooks_d"] = cooks_d
dfm_fit["fitted"] = model.fittedvalues.values

top_influents = dfm_fit.nlargest(8, "cooks_d")[
    ["date_only", "name", "distance_km", "pace_min_km",
     "weather_temperature", "elev_per_km", "pace_last_5_mean",
     "fitted", "resid", "cooks_d"]
]
print(f"Seuil Cook's distance : {seuil:.4f}")
print("\n8 sorties les plus influentes :")
print(top_influents.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
ax.stem(range(len(cooks_d)), cooks_d, basefmt=" ", linefmt="#FC4C02", markerfmt="o")
ax.axhline(seuil, color="black", ls="--", label=f"Seuil 4/n = {seuil:.3f}")
ax.set_xlabel("Index observation")
ax.set_ylabel("Cook's distance")
ax.set_title("Observations influentes")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Résidus par saison

In [ ]:
dfm_fit["month"] = dfm_fit["date_only"].dt.month
dfm_fit["season"] = dfm_fit["month"].map(lambda m:
    "Hiver" if m in [12, 1, 2] else "Printemps" if m in [3, 4, 5]
    else "Été" if m in [6, 7, 8] else "Automne")

print("Résidus par saison :")
print(dfm_fit.groupby("season")["resid"].agg(["count", "mean", "std"]).round(3))

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=dfm_fit, x="season", y="resid",
            order=["Hiver", "Printemps", "Été", "Automne"],
            color="#FC4C02", ax=ax)
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Résidus (min/km)")
ax.set_title("Résidus par saison")
plt.tight_layout()
plt.show()

## 9. Décomposition de la variance expliquée

Combien chaque bloc de variables explique seul ?

In [ ]:
f_meteo    = "pace_min_km ~ weather_temperature + I(weather_temperature**2)"
f_effort   = "pace_min_km ~ distance_km + elev_per_km"
f_forme    = "pace_min_km ~ pace_last_5_mean"
f_me       = f_meteo + " + distance_km + elev_per_km"
f_mf       = f_meteo + " + pace_last_5_mean"
f_ef       = f_effort + " + pace_last_5_mean"

models = {
    "Météo seule (T, T²)":           smf.ols(f_meteo, data=dfm).fit(),
    "Effort seul (dist, D+/km)":     smf.ols(f_effort, data=dfm).fit(),
    "Forme seule (pace_last_5)":     smf.ols(f_forme, data=dfm).fit(),
    "Météo + Effort":                smf.ols(f_me, data=dfm).fit(),
    "Météo + Forme":                 smf.ols(f_mf, data=dfm).fit(),
    "Effort + Forme":                smf.ols(f_ef, data=dfm).fit(),
    "Complet (5 var)":               model,
}

print(f"{'Modèle':<32} {'R²':>7} {'R² adj':>8} {'RMSE s/km':>10} {'AIC':>8}")
print("-" * 72)
for name, m in models.items():
    rmse_s = np.sqrt(m.mse_resid) * 60
    print(f"{name:<32} {m.rsquared:>7.3f} {m.rsquared_adj:>8.3f} {rmse_s:>10.1f} {m.aic:>8.1f}")

## 10. Optimum de température perso : bootstrap

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 1000
n = len(dfm)
optima = []

for b in range(N_BOOT):
    idx = rng.integers(0, n, size=n)
    try:
        m = smf.ols(FORMULA, data=dfm.iloc[idx]).fit()
        b1 = m.params["weather_temperature"]
        b2 = m.params["I(weather_temperature ** 2)"]
        if b2 > 0:
            optima.append(-b1 / (2 * b2))
    except Exception:
        pass

optima = np.array(optima)
b1 = model.params["weather_temperature"]
b2 = model.params["I(weather_temperature ** 2)"]
opt_point = -b1 / (2 * b2)
ic_lo, ic_hi = np.percentile(optima, [2.5, 97.5])

print(f"Bootstrap : {len(optima)}/{N_BOOT} tirages valides (parabole convexe)")
print(f"Optimum ponctuel : {opt_point:.2f} °C")
print(f"Médiane bootstrap : {np.median(optima):.2f} °C")
print(f"IC 95 % (percentile) : [{ic_lo:.2f}, {ic_hi:.2f}] °C")
print(f"\nRéférence El Helou 2012 (marathoniens) : optimum 3.8-9.9 °C")
print("→ Mon optimum perso significativement plus chaud que celui des marathoniens.")

# Figure distribution bootstrap
fig, ax = plt.subplots(figsize=(10, 4))
x_lo = min(0, np.percentile(optima, 1) - 2)
x_hi = np.percentile(optima, 99) + 2
ax.hist(optima, bins=np.linspace(x_lo, x_hi, 60), color="#FC4C02",
        alpha=0.7, edgecolor="black")
ax.axvline(np.median(optima), color="black", lw=2, label=f"Médiane : {np.median(optima):.1f} °C")
ax.axvline(ic_lo, color="black", lw=1, ls="--", label=f"IC 95 %: [{ic_lo:.1f}, {ic_hi:.1f}]")
ax.axvline(ic_hi, color="black", lw=1, ls="--")
ax.axvspan(3.8, 9.9, alpha=0.15, color="blue", label="El Helou 2012 (3.8-9.9 °C)")
ax.set_xlim(x_lo, x_hi)
ax.set_xlabel("Optimum de température (°C)")
ax.set_ylabel("Fréquence (bootstrap)")
ax.set_title(f"Distribution bootstrap de l'optimum perso ({len(optima)} tirages)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 11. Test de robustesse : placebo (pace shufflée)

Si on casse le lien pace↔T°, où se place l'optimum sous H0 ? Permet de confirmer que le signal n'est pas un artefact de la distribution.

In [ ]:
rng2 = np.random.default_rng(123)
N_PLACEBO = 500
optima_h0 = []

for b in range(N_PLACEBO):
    sample = dfm.copy()
    sample["pace_min_km"] = rng2.permutation(sample["pace_min_km"].values)
    try:
        m = smf.ols(FORMULA, data=sample).fit()
        b1h = m.params["weather_temperature"]
        b2h = m.params["I(weather_temperature ** 2)"]
        if b2h > 0:
            optima_h0.append(-b1h / (2 * b2h))
    except Exception:
        pass

optima_h0 = np.array(optima_h0)
print(f"=== PLACEBO : pace shufflée ===")
print(f"{len(optima_h0)}/{N_PLACEBO} tirages valides (parabole convexe)")
print(f"Optimum réel : {opt_point:.2f} °C, IC [{ic_lo:.1f}, {ic_hi:.1f}]")
print(f"Optimum placebo médiane : {np.median(optima_h0):.2f} °C")
print(f"Optimum placebo IC 95 % : [{np.percentile(optima_h0, 2.5):.1f}, {np.percentile(optima_h0, 97.5):.1f}] °C")
print("→ Bootstrap réel bien plus concentré que placebo : signal réel confirmé.")

# Test F : quadratique vs linéaire
f_lin = "pace_min_km ~ weather_temperature + distance_km + elev_per_km + pace_last_5_mean"
model_lin = smf.ols(f_lin, data=dfm).fit()
rss_lin = (model_lin.resid ** 2).sum()
rss_quad = (model.resid ** 2).sum()
df_den = len(dfm) - 6
f_stat = ((rss_lin - rss_quad) / 1) / (rss_quad / df_den)
p_val = 1 - f_dist.cdf(f_stat, 1, df_den)
print(f"\n=== Test F quadratique vs linéaire ===")
print(f"Linéaire (sans T²) : R² adj = {model_lin.rsquared_adj:.3f}, AIC = {model_lin.aic:.1f}")
print(f"Quadratique        : R² adj = {model.rsquared_adj:.3f}, AIC = {model.aic:.1f}")
print(f"F = {f_stat:.3f}, p = {p_val:.4f}")

## 12. Figure maîtresse : pace vs T° avec IC bootstrap

In [ ]:
# Grille de T° pour prédiction, autres variables fixées à leur médiane
T_grid = np.linspace(dfm["weather_temperature"].min(),
                     dfm["weather_temperature"].max(), 100)
med_dist = dfm["distance_km"].median()
med_elev = dfm["elev_per_km"].median()
med_pace5 = dfm["pace_last_5_mean"].median()

N_CURVES = 500
curves = np.zeros((N_CURVES, len(T_grid)))
rng3 = np.random.default_rng(2026)
for b in range(N_CURVES):
    idx = rng3.integers(0, len(dfm), size=len(dfm))
    try:
        m = smf.ols(FORMULA, data=dfm.iloc[idx]).fit()
        pred = (m.params["Intercept"]
                + m.params["weather_temperature"] * T_grid
                + m.params["I(weather_temperature ** 2)"] * T_grid**2
                + m.params["distance_km"] * med_dist
                + m.params["elev_per_km"] * med_elev
                + m.params["pace_last_5_mean"] * med_pace5)
        curves[b] = pred
    except Exception:
        curves[b] = np.nan

lo = np.nanpercentile(curves, 2.5, axis=0)
hi = np.nanpercentile(curves, 97.5, axis=0)
med_curve = np.nanmedian(curves, axis=0)

fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(dfm["weather_temperature"], dfm["pace_min_km"],
                c=dfm["distance_km"], cmap="viridis", alpha=0.6, s=35,
                edgecolor="white", linewidth=0.3)
plt.colorbar(sc, ax=ax, label="Distance (km)")
ax.fill_between(T_grid, lo, hi, color="#FC4C02", alpha=0.2, label="IC bootstrap 95 %")
ax.plot(T_grid, med_curve, color="#FC4C02", lw=2.5, label="Pace prédite (médiane bootstrap)")
ax.axvline(opt_point, color="black", lw=1.5, ls="--",
           label=f"Optimum perso : {opt_point:.1f} °C [IC: {ic_lo:.1f}, {ic_hi:.1f}]")
ax.axvspan(3.8, 9.9, alpha=0.1, color="blue", label="El Helou 2012 (marathoniens)")
ax.set_xlabel("Température (°C)")
ax.set_ylabel("Pace (min/km)")
ax.set_title(f"Pace vs température — modèle final à conditions médianes\n"
             f"(n = {len(dfm)}, R² ajusté = {model.rsquared_adj:.3f})")
ax.legend(loc="upper left")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

import os
os.makedirs("../docs/figures", exist_ok=True)
fig.savefig("../docs/figures/07_pace_vs_T_final.png", dpi=150, bbox_inches="tight")
print("Figure sauvegardée : docs/figures/07_pace_vs_T_final.png")

## 13. Effet de la T° quantifié

In [ ]:
def pace_at_T(T):
    return (model.params["Intercept"]
            + model.params["weather_temperature"] * T
            + model.params["I(weather_temperature ** 2)"] * T**2
            + model.params["distance_km"] * med_dist
            + model.params["elev_per_km"] * med_elev
            + model.params["pace_last_5_mean"] * med_pace5)

pace_opt = pace_at_T(opt_point)
rows = []
for T in [0, 5, 10, 15, 20, 25, 30]:
    p = pace_at_T(T)
    delta_sec = (p - pace_opt) * 60
    rows.append({"T (°C)": T, "pace (min/km)": f"{p:.2f}",
                 "pace (min:ss)": f"{int(p)}:{int(round((p - int(p))*60)):02d}",
                 "écart vs optimum (s/km)": f"+{delta_sec:.0f}"})
print(f"Effet T° à conditions médianes (optimum {opt_point:.1f} °C) :")
print(pd.DataFrame(rows).to_string(index=False))

print("\nSensibilité locale (dérivée dpace/dT°) :")
for T in [5, 10, 15, 20, 25]:
    deriv = b1 + 2 * b2 * T
    print(f"  à {T} °C : {deriv*60:+.1f} s/km par °C")

## 14. Conclusions

### Modèle final retenu
- 5 variables : T° + T² + distance + D+/km + pace_last_5_mean.
- R² ajusté ≈ 0.23, RMSE ≈ 42 s/km, Durbin-Watson ≈ 1.82 (autocorrélation négligeable).
- Tous les coefficients significatifs à p < 0.05.

### Résultats principaux
- **Forme physique dominante** : pace_last_5_mean apporte +60 % de R² ajusté par rapport au modèle sans forme. Les volumes glissants (7, 14, 28, 90 j), ACWR et monotonie n'apportent rien ; la pace moyenne récente est la bonne spécification.
- **Optimum perso ≈ 16-17 °C** (IC bootstrap resserré), significativement plus chaud que celui des marathoniens d'El Helou (3.8-9.9 °C). Cohérent avec un profil d'endurance.
- **Météo et effort orthogonaux** : chacun ~9 % de variance seul, 17 % ensemble.
- **Humidité, days_since_last*, volumes glissants** : non prédictifs à n=121 une fois la forme contrôlée.
- **Pas de biais saisonnier résiduel** après contrôle T° + forme.

### Limites
- Variance résiduelle encore ≈ 77 % : forme du jour, facteurs sociaux (sorties en famille/couple), fatigue aiguë non capturés.
- Résidus légèrement asymétriques (queue droite) : sorties atypiques (fractionné court, trail D+ extrême) mal prédites.
- FC exclue du modèle principal (13/131 sorties avec FC, pré/post-acquisition Garmin). À réintégrer sur sous-échantillon post-Garmin dans un modèle complémentaire.

### Prochaines étapes
- Modèle mixte par-km (notebook 09) : décomposition de variance between/within pour répondre à la question TZ enregistrée sur l'allure instantanée et l'état d'avancement.
- Classification pacing (Pryor) : séances dédiées après la soutenance intermédiaire.